# Switchboard (MS98) statistics, organised by the RSML disfluency taxonomy

A statistics-only notebook: no models, no training. It reads two things.

1. **The raw ISIP / Mississippi State release** (`switchboard_word_alignments.tar.gz`, read in place without
   unpacking). Everything about the corpus itself comes from here: sizes, splits, **the special token types
   of the markup** (silence, noises, laughter, asides, partial words, mispronunciations, coinages,
   pronunciation variants), vocabulary, lexicon and phones.
2. **The project's labelled dataset** (`data/processed/swb_tokens.jsonl.gz`): one label per token, produced by
   rules that map Switchboard onto the **RSML** tag set (BhashaCheck `rsml 3.3.5`, the `[tags]` section:
   hesitations, disfluency spans, isolated paralinguistics, isolated other, paralinguistic spans, prosody spans).
   Every disfluency statistic below uses these labels, grouped by RSML tag or notation.
Mispronunciations use `[spoken](intended)`; this notation is reported separately from the `[tags]` inventory.

**Provenance matters.** Switchboard carries no RSML annotation. A label is **gold** when it is read straight
from explicit ISIP markup (partial words, mispronunciations, noises, hesitation lexemes) and **silver** when a
rule derives it (repetition, repair, false start, prolongation, discourse-marker filler, short-turn
backchannels). Silver labels are heuristics, not human annotation; each table says which kind it is.
RSML tags that Switchboard never marks (stutter blocks, yawns, pitch, ...) are listed with count 0 rather than
dropped.

| Section | Content |
|---|---|
| 1 | Load and parse the raw archive |
| 2 | Corpus overview and splits |
| 3 | **Special token types** of the markup |
| 4 | Utterances, sides, speaking rate |
| 5 | Vocabulary, lexicon and phones |
| 6 | **The RSML taxonomy** and how the dataset covers it |
| 7 | **Full statistics per RSML category** |
| 8 | Across categories: splits, speakers, position, neighbours |
| 9 | Timing: pauses and durations per category |
| 10 | The fluent skeleton: what removing the disfluencies leaves |
| 11 | Headline numbers |

Run the same analysis from the repository root with:

```bash
.venv/bin/python scripts/switchboard_stats.py --out results/switchboard_stats
```

The script reads this notebook as its source of rules and exports CSV tables and PNG plots.


In [ ]:
# On Colab, uncomment:
# %pip install -q pandas numpy scipy matplotlib huggingface_hub

import re
from collections import Counter
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)


def _fmt(v):
    """Whole numbers without decimals, shares with 3 decimals, everything else with 2."""
    if v != v:
        return "-"
    if v in (float("inf"), float("-inf")):
        return "inf"
    if v == round(v):
        return f"{v:,.0f}"
    return f"{v:.3f}" if abs(v) < 1 else f"{v:,.2f}"


pd.set_option("display.float_format", _fmt)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10,
    "font.size": 10, "legend.frameon": False,
})
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = "#2f5fb3", "#e08a1e", "#2f9e72", "#b84a3c", "#7a5ba6", "#8a8f98"


def pct(x, d=2):
    return f"{100 * x:.{d}f}%"

## 1. Load and parse the archive

The archive is looked up in `data/raw/switchboard_ms98/` (local checkout), then in Colab's
`/content`, and finally downloaded from the private Hugging Face dataset repo that holds the
project data (needs `hf auth login` or `HF_TOKEN`).

In [ ]:
RAW_NAME = "switchboard_word_alignments.tar.gz"
HF_REPO = "imradhe/cl2-disfluency-restoration-data"


def find_archive() -> Path:
    for base in (Path("data/raw/switchboard_ms98"), Path("../data/raw/switchboard_ms98"),
                 Path("/content/data/raw/switchboard_ms98")):
        if (base / RAW_NAME).exists():
            return base / RAW_NAME
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_REPO, f"raw/switchboard_ms98/{RAW_NAME}", repo_type="dataset"))


ARCHIVE = find_archive()
print(f"{ARCHIVE}  ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# One pass over the archive. Tokens are stored as integer codes (38k distinct strings, 4M rows).
TOKEN_CODE, TOKEN_LIST = {}, []


def code_of(tok):
    c = TOKEN_CODE.get(tok)
    if c is None:
        c = TOKEN_CODE[tok] = len(TOKEN_LIST)
        TOKEN_LIST.append(tok)
    return c


chunks, lexicon_lines, n_trans_files = [], None, 0
click_context_rows = []
with tarfile.open(ARCHIVE, "r:gz") as tf:
    for member in tf:
        if not member.isfile():
            continue
        name = member.name.rsplit("/", 1)[-1]
        if name.endswith("-ms98-a-word.text"):
            utt, t0, t1, tok = [], [], [], []
            for line in tf.extractfile(member).read().decode("utf8").splitlines():
                p = line.split()
                if len(p) < 4:
                    continue
                utt.append(int(p[0].rsplit("-", 1)[1]))
                t0.append(float(p[1]))
                t1.append(float(p[2]))
                tok.append(code_of(p[3]))
            chunks.append((name.split("-")[0], np.array(utt, np.int32), np.array(t0), np.array(t1),
                           np.array(tok, np.int32)))
        elif name.endswith("-ms98-a-trans.text"):
            n_trans_files += 1
            for line in tf.extractfile(member).read().decode("utf8").splitlines():
                p = line.split(maxsplit=3)
                if len(p) == 4 and re.search(r"\b(?:click|clicks|clicking|clicked|tsk|tut|smack|cluck)\b", p[3], re.I):
                    click_context_rows.append({"archive file": member.name, "utterance": p[0], "transcript": p[3].rstrip()})
        elif name == "sw-ms98-dict.text":
            lexicon_lines = tf.extractfile(member).read().decode("utf8").splitlines()

chunks.sort(key=lambda c: c[0])
SIDES = [c[0] for c in chunks]
side = np.concatenate([np.full(len(c[1]), i, np.int32) for i, c in enumerate(chunks)])
utt = np.concatenate([c[1] for c in chunks])
start = np.concatenate([c[2] for c in chunks])
end = np.concatenate([c[3] for c in chunks])
tok = np.concatenate([c[4] for c in chunks])
g = side.astype(np.int64) * 1_000_000 + utt          # unique utterance key
del chunks
print(f"{len(SIDES):,} speaker sides, {len(tok):,} word-level rows, {len(TOKEN_LIST):,} distinct token strings, "
      f"{n_trans_files:,} utterance-level files, lexicon lines: {len(lexicon_lines):,}")

### Token classes

Every row of the word files is assigned one class from its markup:

| Class | Markup | Example |
|---|---|---|
| `SILENCE` | `[silence]` | |
| `NOISE` | `[noise]`, `[vocalized-noise]` | |
| `LAUGHTER` | `[laughter]` | |
| `ASIDE` | `<b_aside>`, `<e_aside>` | |
| `WORD` | ordinary word, incl. `[laughter-word]` (spoken while laughing) and `word_1` pronunciation variants | `uh`, `[laughter-yeah]`, `them_1` |
| `PARTIAL` | word fragment, `x-` (spoken part) with optional `[completion]` | `i-`, `th[e]-`, `-[o]kay` |
| `MISPRON` | `[spoken/intended]` | `[abrode/abroad]` |
| `COINAGE` | `{word}` | `{kiddos}` |

**Lexical tokens** are `WORD + PARTIAL + MISPRON + COINAGE`; all per-token rates below use them as
the denominator. Silences, noises, laughter and asides are transparent: they are skipped when looking for
neighbouring words (so `and [noise] and` counts as a repeat).

In [ ]:
CLASSES = ["SILENCE", "NOISE", "LAUGHTER", "ASIDE", "WORD", "PARTIAL", "MISPRON", "COINAGE"]
SIL, NOI, LAU, ASI, WRD, PAR, MIS, COI = range(8)
LEXICAL = [WRD, PAR, MIS, COI]
_LAUGH = re.compile(r"^\[laughter-(.+)\]$")


def classify(t):
    """-> (class id, identity string, spoken-while-laughing flag)."""
    if t == "[silence]":
        return SIL, t, False
    if t in ("[noise]", "[vocalized-noise]"):
        return NOI, t, False
    if t == "[laughter]":
        return LAU, t, False
    if t in ("<b_aside>", "<e_aside>"):
        return ASI, t, False
    m = _LAUGH.match(t)
    inner, laugh = (m.group(1), True) if m else (t, False)
    low = inner.lower()
    if inner.startswith("{") and inner.endswith("}"):
        return COI, low, laugh
    if inner.startswith("[") and "/" in inner and inner.endswith("]"):
        return MIS, low, laugh
    if inner.endswith("-") or inner.startswith("-"):
        return PAR, low, laugh
    return WRD, re.sub(r"_\d+", "", low), laugh       # identity: lower-case, pronunciation-variant suffix removed


IDENT_ID, IDENT_LIST = {}, []
TOK_CLS = np.zeros(len(TOKEN_LIST), np.int8)
TOK_LAUGH = np.zeros(len(TOKEN_LIST), bool)
TOK_IDENT = np.zeros(len(TOKEN_LIST), np.int32)
for code, t in enumerate(TOKEN_LIST):
    c, ident_str, laugh = classify(t)
    if ident_str not in IDENT_ID:
        IDENT_ID[ident_str] = len(IDENT_LIST)
        IDENT_LIST.append(ident_str)
    TOK_CLS[code], TOK_LAUGH[code], TOK_IDENT[code] = c, laugh, IDENT_ID[ident_str]

cls = TOK_CLS[tok]                 # class id per row
ident = TOK_IDENT[tok]             # identity id per row
is_lex = np.isin(cls, LEXICAL)
dur = end - start

conv = np.array([int(s[2:6]) for s in SIDES])                 # conversation number of each side
# Split by conversation (Charniak & Johnson convention, as used in the earlier project work)
split_of_side = np.where((conv >= 4010) & (conv <= 4153), "test",
                         np.where((conv >= 4519) & (conv <= 4936), "dev", "train"))
split_row = split_of_side[side]
print("class counts:", dict(zip(CLASSES, np.bincount(cls, minlength=8))))

## 2. Corpus overview

In [ ]:
side_dur = pd.Series(end).groupby(side).max().to_numpy()        # seconds of each side
speech_time = dur[(cls != SIL)].clip(min=0).sum()
silence_time = dur[(cls == SIL)].clip(min=0).sum()
n_lex = int(is_lex.sum())
n_utt = len(np.unique(g))

overview = pd.DataFrame({"value": {
    "conversations": len(set(conv)),
    "speaker sides": len(SIDES),
    "utterance segments (incl. silence-only)": n_utt,
    "word-level rows": len(tok),
    "lexical tokens (denominator for all rates)": n_lex,
    "distinct token strings": len(TOKEN_LIST),
    "hours of conversation audio": side_dur.sum() / 3600 / 2,
    "hours summed over both sides": side_dur.sum() / 3600,
    "mean side length (minutes)": side_dur.mean() / 60,
    "share of side time that is [silence]*": silence_time / (speech_time + silence_time),
    "rows with end <= start (alignment glitches)": int((dur <= 0).sum()),
}})
print("* [silence] rows also cover stretches where the other speaker is talking; this is not a pause rate.")
overview

### Train / dev / test split

By conversation id: **test** = sw4010 to sw4153, **dev** = sw4519 to sw4936, **train** = everything else.

In [ ]:
def split_table():
    rows = {}
    for name in ["train", "dev", "test"]:
        sm = split_of_side == name
        rm = split_row == name
        rows[name] = {
            "conversations": len(set(conv[sm])),
            "sides": int(sm.sum()),
            "utterance segments": len(np.unique(g[rm])),
            "lexical tokens": int((is_lex & rm).sum()),
            "hours": side_dur[sm].sum() / 3600,
        }
    t = pd.DataFrame(rows).T
    t.loc["total"] = t.sum()
    t["share of tokens"] = t["lexical tokens"] / t.loc["total", "lexical tokens"]
    return t


splits = split_table()
splits

### Token classes

In [ ]:
counts = np.bincount(cls, minlength=8)
tab = pd.DataFrame({"rows": counts}, index=CLASSES)
tab["share of all rows"] = tab.rows / tab.rows.sum()
tab["per 1,000 lexical tokens"] = tab.rows / n_lex * 1000
tab["spoken while laughing"] = [int((TOK_LAUGH[tok] & (cls == c)).sum()) if c in LEXICAL else 0 for c in range(8)]
tab

In [ ]:
show = tab.drop(index=["SILENCE", "WORD"])
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(show.index[::-1], show["per 1,000 lexical tokens"][::-1], color=BLUE)
for i, v in enumerate(show["per 1,000 lexical tokens"][::-1]):
    ax.text(v, i, f" {v:.1f}", va="center", fontsize=9)
ax.set_xlabel("rows per 1,000 lexical tokens")
ax.set_title("Non-word markup in the transcripts")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 3. Special token types of the markup

Every distinct kind of non-ordinary token in the word files, counted exhaustively. A row is one *word-level
alignment row*. `[laughter-word]` rows are ordinary words spoken while laughing, so they also carry a word.

In [ ]:
BUCKETS = ["[silence]", "[noise]", "[vocalized-noise]", "[laughter]",
           "[laughter-word]  (word spoken while laughing)",
           "<b_aside> / <e_aside>  (aside markers)",
           "partial word with annotated completion, th[e]-",
           "partial word, bare fragment, i-",
           "partial word, word-final fragment, -[o]kay",
           "mispronunciation, [spoken/intended]",
           "coinage, {word}",
           "pronunciation variant, word_1",
           "ordinary word"]
B = {b: i for i, b in enumerate(BUCKETS)}


def bucket_of(t):
    if t == "[silence]":
        return B["[silence]"]
    if t == "[noise]":
        return B["[noise]"]
    if t == "[vocalized-noise]":
        return B["[vocalized-noise]"]
    if t == "[laughter]":
        return B["[laughter]"]
    if t in ("<b_aside>", "<e_aside>"):
        return B["<b_aside> / <e_aside>  (aside markers)"]
    if _LAUGH.match(t):
        return B["[laughter-word]  (word spoken while laughing)"]
    if t.startswith("{") and t.endswith("}"):
        return B["coinage, {word}"]
    if t.startswith("[") and "/" in t and t.endswith("]"):
        return B["mispronunciation, [spoken/intended]"]
    if t.startswith("-"):
        return B["partial word, word-final fragment, -[o]kay"]
    if t.endswith("-"):
        return B["partial word with annotated completion, th[e]-"] if "[" in t else B["partial word, bare fragment, i-"]
    if re.search(r"_\d+", t):
        return B["pronunciation variant, word_1"]
    return B["ordinary word"]


TOK_BUCKET = np.array([bucket_of(t) for t in TOKEN_LIST], np.int8)
bucket_row = TOK_BUCKET[tok]
nb = len(BUCKETS)
bcount = np.bincount(bucket_row, minlength=nb)
rows = {}
for b, name in enumerate(BUCKETS):
    m = bucket_row == b
    r = {"rows": int(bcount[b]), "per 1,000 lexical tokens": bcount[b] / n_lex * 1000,
         "distinct strings": int((TOK_BUCKET == b).sum()),
         "median duration (ms)": float(np.median(dur[m].clip(min=0)) * 1000) if m.any() else np.nan}
    for sp in ("train", "dev", "test"):
        r[f"per 1,000 tokens, {sp}"] = (m & (split_row == sp)).sum() / (is_lex & (split_row == sp)).sum() * 1000
    rows[name] = r
special = pd.DataFrame(rows).T
special

In [ ]:
# the most frequent strings of every special bucket
tc = np.bincount(tok, minlength=len(TOKEN_LIST))
pd.Series({
    name: ",  ".join(f"{TOKEN_LIST[c]} ({tc[c]:,})"
                     for c in sorted(np.flatnonzero(TOK_BUCKET == b), key=lambda c: -tc[c])[:6])
    for b, name in enumerate(BUCKETS) if b not in (B["ordinary word"], B["[silence]"], B["[noise]"],
                                                    B["[vocalized-noise]"], B["[laughter]"])
}, name="most frequent strings").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
show = special.drop(index=["[silence]", "ordinary word"])
vals = show["per 1,000 lexical tokens"][::-1]
ax.barh([s.split("  (")[0] for s in vals.index], vals.to_numpy(), color=BLUE)
for i, v in enumerate(vals.to_numpy()):
    ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("rows per 1,000 lexical tokens (log scale)")
ax.set_title("Special token types in the transcripts")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### Audit for `@tsk`: click markers versus ordinary words

Search the **raw token strings**, not just the processed hesitation labels. Explicit bracketed click/tsk
markers would support `@tsk`. Plain `click`, `clicks`, `clicked`, `clicking`, `clicker`, `smack`, and `smacking`
are lexical words, not event tags. Generic `[noise]` / `[vocalized-noise]` might contain a click, but their
annotation does not tell us which sound occurred, so they are not assigned to `@tsk`.
The table keeps absent candidate tags visible with a zero count; the context table makes lexical hits auditable.


In [ ]:
TSK_MARKERS = ["[click]", "[clicks]", "[tongue-click]", "[tongue_click]", "[tongue-clicks]",
               "[dental-click]", "[mouth-click]", "[tsk]", "[tsk-tsk]", "[tut]", "[tut-tut]"]
TSK_LEXEMES = ["tsk", "tsk-tsk", "tut", "tut-tut"]
raw_form_counts = Counter()
for raw, count in zip(TOKEN_LIST, tc):
    raw_form_counts[raw.lower()] += int(count)
n_tsk_markers = sum(raw_form_counts[t] for t in TSK_MARKERS)
lexical_clicks = ["click", "clicks", "clicked", "clicking", "clicker", "smack", "smacking", "cluck"]
audit_rows = []
for forms, evidence in [(TSK_MARKERS, "explicit event marker: usable for @tsk"),
                        (TSK_LEXEMES, "candidate lexeme: requires hesitation label"),
                        (lexical_clicks, "ordinary lexical word: excluded"),
                        (["[noise]", "[vocalized-noise]"], "generic noise: sound is unspecified")]:
    for form in forms:
        audit_rows.append({"raw token": form, "rows": raw_form_counts[form], "evidence": evidence})
tsk_audit = pd.DataFrame(audit_rows)
print(f"Explicit click/tsk event markers: {n_tsk_markers:,}. "
      f"Candidate tsk/tut lexemes: {sum(raw_form_counts[t] for t in TSK_LEXEMES):,}. "
      "Lexical click/smack words and generic noise markers are not counted as @tsk.")
tsk_audit


In [ ]:
click_contexts = pd.DataFrame(click_context_rows, columns=["archive file", "utterance", "transcript"])
pd.set_option("display.max_colwidth", 180)
click_contexts


## 4. Utterances, sides and speaking rate

In [ ]:
first_row = np.r_[True, g[1:] != g[:-1]]
ustart = np.flatnonzero(first_row)
usize = np.diff(np.r_[ustart, len(g)])
utt_key = g[ustart]
utt_lex = np.add.reduceat(is_lex.astype(np.int32), ustart)         # lexical tokens per utterance
utt_t0 = np.minimum.reduceat(start, ustart)
utt_t1 = np.maximum.reduceat(end, ustart)
utt_split = split_row[ustart]

has_speech = utt_lex > 0
print(f"{n_utt:,} utterance segments; {(~has_speech).sum():,} ({pct((~has_speech).mean(), 1)}) contain no lexical token "
      f"(silence / noise / laughter only)")

len_tab = pd.Series(utt_lex[has_speech]).describe(percentiles=[.05, .25, .5, .75, .95]).to_frame("lexical tokens per utterance")
len_tab.loc["utterance duration median (s)"] = np.median((utt_t1 - utt_t0)[has_speech])
len_tab

In [ ]:
# Speaking rate: lexical tokens per second of non-silent alignment time, per side
nonsil_dur = np.bincount(side[cls != SIL], weights=dur[cls != SIL].clip(min=0), minlength=len(SIDES))
lex_per_side = np.bincount(side[is_lex], minlength=len(SIDES))
rate = lex_per_side / nonsil_dur

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(utt_lex[has_speech], bins=np.arange(0, 61), color=BLUE)
axes[0].set_xlabel("lexical tokens per utterance")
axes[0].set_ylabel("utterances")
axes[0].set_title("Utterance length")
axes[1].hist(side_dur / 60, bins=30, color=GREEN)
axes[1].set_xlabel("minutes per side")
axes[1].set_title("Side length")
axes[2].hist(rate, bins=30, color=ORANGE)
axes[2].set_xlabel("lexical tokens per second of non-silent time")
axes[2].set_title("Speaking rate per side")
plt.tight_layout()
plt.show()
print(f"speaking rate: median {np.median(rate):.2f} tokens/s, IQR {np.percentile(rate, 25):.2f} to {np.percentile(rate, 75):.2f}; "
      f"median side {np.median(side_dur) / 60:.1f} min, {int(np.median(lex_per_side)):,} lexical tokens")
near5 = ((side_dur / 60 > 4.9) & (side_dur / 60 < 5.1)).mean()
near10 = ((side_dur / 60 > 9.9) & (side_dur / 60 < 10.1)).mean()
print(f"{pct(near5, 1)} of sides are 5.0 +/- 0.1 min and {pct(near10, 1)} are 10.0 +/- 0.1 min: most sides are cut to fixed-length excerpts")

## 5. Vocabulary and lexicon coverage

In [ ]:
word_mask = cls == WRD
wc = np.bincount(ident[word_mask], minlength=len(IDENT_LIST))
vocab = pd.Series(wc, index=IDENT_LIST)
vocab = vocab[vocab > 0].sort_values(ascending=False)
n_word_tok = int(vocab.sum())
cum = vocab.cumsum() / n_word_tok

vt = pd.Series({
    "word tokens (class WORD, lower-cased)": n_word_tok,
    "word types": len(vocab),
    "types seen once (hapax)": int((vocab == 1).sum()),
    "hapax share of types": (vocab == 1).mean(),
    "types seen 5+ times": int((vocab >= 5).sum()),
    "tokens covered by top 10 types": cum.iloc[9],
    "tokens covered by top 100 types": cum.iloc[99],
    "tokens covered by top 1,000 types": cum.iloc[999],
    "tokens covered by top 10,000 types": cum.iloc[9999],
}).to_frame("value")
vt

In [ ]:
ranks = np.arange(1, len(vocab) + 1)
sel = (ranks >= 10) & (ranks <= 10_000)
slope, intercept = np.polyfit(np.log10(ranks[sel]), np.log10(vocab.to_numpy()[sel]), 1)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].loglog(ranks, vocab.to_numpy(), color=BLUE, lw=1.4)
axes[0].set_xlabel("rank")
axes[0].set_ylabel("frequency")
axes[0].set_title(f"Zipf plot (slope {slope:.2f} on ranks 10 to 10,000)")
axes[1].semilogx(ranks, cum.to_numpy(), color=GREEN, lw=1.6)
axes[1].set_xlabel("vocabulary size (most frequent first)")
axes[1].set_ylabel("share of word tokens covered")
axes[1].set_title("Coverage curve")
top = vocab.head(25)[::-1]
axes[2].barh(top.index, top.to_numpy() / 1000, color=ORANGE)
axes[2].set_xlabel("thousand tokens")
axes[2].set_title("25 most frequent words")
axes[2].grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### Pronunciation lexicon and phones

`sw-ms98-dict.text` is organised in sections (partial words, words containing laughter, alternate
pronunciations, **hesitations**, proper nouns, anomalous words, coinages, normal items). The section
headers are used below to read the lexicon's own list of hesitation sounds.

In [ ]:
SECTIONS = {"partial word pronunciations": "partial", "words containing laughter": "laughter",
            "common alternate pronunciations": "alternate", "hesitations": "hesitations",
            "proper nouns": "proper nouns", "anomalous words": "anomalous", "coinages": "coinages",
            "normal lexical items": "normal"}
LEXICON, SECTION_OF, section, n_lines = {}, {}, "?", 0
for i, line in enumerate(lexicon_lines):
    s = line.strip()
    if not s or (i == 0 and s.startswith("file:")):
        continue
    if s.startswith("#"):
        key = s.lstrip("# ").rstrip(":").strip().lower()
        if key in SECTIONS:
            section = SECTIONS[key]
        continue
    word, *phones = s.split()
    n_lines += 1
    if phones not in LEXICON.setdefault(word, []):      # some entries are listed twice (e.g. hesitations)
        LEXICON[word].append(phones)
    SECTION_OF.setdefault(word, section)

PHONES = sorted({p for prons in LEXICON.values() for pr in prons for p in pr})
VOWELS = {"aa", "ae", "ah", "ao", "aw", "ax", "ay", "eh", "er", "ey", "ih", "iy", "ow", "oy", "uh", "uw"}
lex_summary = pd.Series({
    "lexicon entries (distinct strings)": len(LEXICON),
    "entry lines in the file": n_lines,
    "distinct (word, pronunciation) pairs": sum(len(v) for v in LEXICON.values()),
    "entries with 2+ distinct pronunciations": sum(len(v) > 1 for v in LEXICON.values()),
    "phone inventory size": len(PHONES),
    "of which treated as vowels (see VOWELS)": len([p for p in PHONES if p in VOWELS]),
}).to_frame("value")
print("phones:", " ".join(PHONES))
print("entries per lexicon section:", pd.Series(SECTION_OF).value_counts().to_dict())
lex_summary

In [ ]:
def lookup(t):
    """Raw string first, then lower-case; None if the lexicon has no entry."""
    if t in LEXICON:
        return LEXICON[t]
    if t.lower() in LEXICON:
        return LEXICON[t.lower()]
    return None


type_counts = np.bincount(tok, minlength=len(TOKEN_LIST))
cov = {c: [0, 0, 0, 0] for c in LEXICAL}          # token hit, token total, type hit, type total
misses = []
for code, t in enumerate(TOKEN_LIST):
    c = TOK_CLS[code]
    if c not in LEXICAL:
        continue
    hit = lookup(t) is not None
    cov[c][1] += type_counts[code]
    cov[c][3] += 1
    if hit:
        cov[c][0] += type_counts[code]
        cov[c][2] += 1
    else:
        misses.append((t, int(type_counts[code])))
cov_tab = pd.DataFrame({CLASSES[c]: {"token coverage": v[0] / v[1], "type coverage": v[2] / v[3], "tokens": v[1]}
                        for c, v in cov.items()}).T
print(f"{len(misses)} token strings have no lexicon entry" + (f", e.g. {sorted(misses, key=lambda x: -x[1])[:8]}" if misses else ""))
cov_tab

In [ ]:
# Phone frequencies in running speech (first listed pronunciation of every lexical token)
phone_counts = dict.fromkeys(PHONES, 0)
pron_len_sum, pron_n = 0, 0
for code, t in enumerate(TOKEN_LIST):
    if TOK_CLS[code] not in LEXICAL:
        continue
    prons = lookup(t)
    if not prons:
        continue
    n = int(type_counts[code])
    for p in prons[0]:
        phone_counts[p] += n
    pron_len_sum += len(prons[0]) * n
    pron_n += n
pc = pd.Series(phone_counts).sort_values(ascending=False)
colors = [ORANGE if p in VOWELS else BLUE for p in pc.index]

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(pc.index, pc.to_numpy() / pc.sum() * 100, color=colors)
ax.set_ylabel("% of phone tokens")
ax.set_title("Phone frequency in running speech (orange = vowels)")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
vshare = pc[[p for p in pc.index if p in VOWELS]].sum() / pc.sum()
print(f"{pc.sum():,} phone tokens; mean {pron_len_sum / pron_n:.2f} phones per lexical token; vowels {pct(vshare, 1)} of phone tokens")

## 6. The RSML taxonomy and how the dataset covers it

The taxonomy below is the `[tags]` section of the RSML header (`rsml 3.3.5`), reproduced as data. The labelled
dataset assigns **one primary label per token** with these rules (parameters from the labelling config):

| Dataset label | RSML tag / notation | Provenance | Rule |
|---|---|---|---|
| `FILLER_HESITATION` | hesitations (`@uhh @umm @hmm @ehh @huh ...`) | gold | token is `uh um ah hm huh` (explicit ISIP lexemes) or one of a silver list (`eh er hmm mm umm uhh ehh ugh ooh ohh ho uh-oh tsk`) |
| `FILLER_BACKCHANNEL` | hesitations (`@uh-huh`) | gold + silver | hyphenated `uh-huh um-hum huh-uh hum-um` (gold); `yeah yep yes nope right sure okay exactly really` when the utterance has at most 3 speech tokens (silver) |
| `FILLER_DM` | disfluency span `filler` | silver | discourse markers `you know`, `i mean`, `you see`, `sort of`, `kind of` (not in literal use) and `well like so actually basically anyway` only in utterance-initial position or next to another filler |
| `REPETITION` | disfluency span `repetition` | silver | the earlier copy of an immediately repeated n-gram (n up to 4), fillers allowed in between; the reparandum is labelled, the last copy stays; a short list of intentional reduplications is excluded |
| `REPAIR` | disfluency span `repair` | silver | reparandum before a correction that copies at least half of it, at most 3 tokens apart, editing terms allowed in between; the correction gets `role=repair` |
| `FALSE_START` | disfluency span `false-start` | silver | abandoned fragment before a clause restart, only when a truncation cue (a partial word or unfinished clause) is present |
| `PARTIAL_WORD` | disfluency span `broken-word` | gold | `x-`, `x[yz]-`, `-[x]yz` in the markup |
| `PROLONGATION` | disfluency span `prolongation` | silver | function word whose duration z-score against its own lexicon history exceeds 2.0 and is at least 0.35 s, not phrase-final |
| `NOISE` | isolated paralinguistics (`@laughter`) and other | gold | `[noise]`, `[vocalized-noise]`, `[laughter]` |
| `MISPRONUNCIATION` | `[spoken](intended)` | gold | explicit Switchboard `[spoken/intended]` pair, represented with the revised RSML notation; cut-off pairs retain primary label `PARTIAL_WORD` |

`role` marks tokens that belong to an episode: `reparandum` (the discarded material), `interregnum` (a filler
sitting between a reparandum and its correction) and `repair` (the corrected material, label `O`).
Mispronunciation is an RSML **notation**, not an extra named tag in the `[tags]` header.
The coverage table includes it but distinguishes tag counts from notation counts.
`@tsk` accepts explicit click/tsk markers, if present; ordinary words such as `click` and generic
`[noise]` / `[vocalized-noise]` are not enough to identify a tongue click (see the raw-data audit in section 3).

Tokens labelled `O`, `PROLONGATION` or `MISPRONUNCIATION` stay in the **fluent skeleton**; the rest are removed.

In [ ]:
RSML_TAGS = {
    "hesitations": ["@umm", "@uhh", "@hmm", "@ugh", "@huh", "@tsk", "@uh-huh", "@ehh", "@ooh", "@uhh-oh", "@ohh", "@ho"],
    "disfluencySpans": ["filler", "repetition", "broken-word", "repair", "false-start", "prolongation"],
    "isolatedParalinguistics": ["@laughter", "@cry", "@hum", "@breathe", "@sniff", "@nose-blowing", "@cough", "@sneeze",
                                "@throat-clearing", "@yawn", "@eating-sounds", "@snore", "@groan", "@sigh"],
    "isolatedOther": ["@unintelligible", "@stutter-block", "@short-pause", "@long-pause"],
    "paralinguisticSpans": ["crying", "yelling", "laughing", "singing", "humming", "whistling", "whispering"],
    "prosodySpans": ["emphasis", "falling-pitch", "raising-pitch", "high-pitch", "low-pitch", "fast-speech", "slow-speech"],
}
RSML_MISPRON = "[spoken](intended)"
print({g: len(t) for g, t in RSML_TAGS.items()}, "->", sum(len(t) for t in RSML_TAGS.values()), "tags")

### Load the labelled dataset

Looked up in `data/processed/`, then Colab's `/content`, then the private Hugging Face dataset repo.

In [ ]:
import gzip, json
from array import array
from collections import Counter

PROC_NAME = "swb_tokens.jsonl.gz"


def find_processed() -> Path:
    for base in (Path("data/processed"), Path("../data/processed"), Path("/content/data/processed")):
        if (base / PROC_NAME).exists():
            return base / PROC_NAME
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_REPO, f"processed/{PROC_NAME}", repo_type="dataset"))


PROC = find_processed()
LABELS = ["O", "FILLER_HESITATION", "FILLER_BACKCHANNEL", "FILLER_DM", "REPETITION", "REPAIR", "FALSE_START",
          "PARTIAL_WORD", "PROLONGATION", "MISPRONUNCIATION", "NOISE"]
KINDS = ["WORD", "PARTIAL", "NOISE", "NOISE_WORD", "COINAGE", "MISPRON"]
ROLES = ["", "reparandum", "interregnum", "repair"]
CUTS = ["", "end", "start", "both"]
LAB, KND, ROL, CUT = ({n: i for i, n in enumerate(v)} for v in (LABELS, KINDS, ROLES, CUTS))
GAP_LABELS = ["FILLER_HESITATION", "FILLER_BACKCHANNEL", "FILLER_DM", "REPETITION", "REPAIR", "FALSE_START", "PARTIAL_WORD"]
FILLERS = ["FILLER_HESITATION", "FILLER_BACKCHANNEL", "FILLER_DM"]

a_lab, a_knd, a_rol, a_cut, a_prol, a_uf, a_ul, a_nph, a_brk = (bytearray() for _ in range(9))
a_t0, a_t1, a_pb = array("d"), array("d"), array("d")
a_ug, a_sd, a_frm = array("q"), array("i"), array("i")
FORM_CODE, FORM_LIST, intended_of = {}, [], {}
mispron_pairs = Counter()
D_SIDE, D_CONV, ucount = [], [], 0

with gzip.open(PROC, "rt") as f:
    for si, line in enumerate(f):
        d = json.loads(line)
        D_SIDE.append(d["sid"])
        D_CONV.append(int(d["conv"]))
        prev = None
        for t in d["tokens"]:
            if t["utt"] != prev:
                ucount += 1
                prev = t["utt"]
            if t["kind"] == "MISPRON":
                mispron_pairs[(t["raw"], (t["spoken"] or "").lower(),
                               (t["intended"] or "").lower(), t["label"])] += 1
            a_lab.append(LAB[t["label"]]); a_knd.append(KND[t["kind"]]); a_rol.append(ROL[t["role"]])
            a_cut.append(CUT[t["cut"]]); a_prol.append(1 if t["prolonged"] else 0)
            a_uf.append(1 if t["utt_first"] else 0); a_ul.append(1 if t["utt_last"] else 0)
            a_brk.append(1 if "[" in t["raw"] else 0)
            ph = t["phones"]
            a_nph.append(min(len(ph), 255) if ph else 0)
            a_t0.append(t["start"] if t["start"] is not None else float("nan"))
            a_t1.append(t["end"] if t["end"] is not None else float("nan"))
            a_pb.append(t["pause_before"] or 0.0)
            a_ug.append(ucount); a_sd.append(si)
            fs = (t["spoken"] or t["raw"]).lower()
            fc = FORM_CODE.get(fs)
            if fc is None:
                fc = FORM_CODE[fs] = len(FORM_LIST)
                FORM_LIST.append(fs)
            a_frm.append(fc)
            if t["kind"] in ("PARTIAL", "MISPRON"):
                intended_of[len(a_lab) - 1] = t["intended"].lower()

lab, knd, rol, cut_t = (np.frombuffer(a, np.uint8) for a in (a_lab, a_knd, a_rol, a_cut))
prolonged, uf, ul, nph = (np.frombuffer(a, np.uint8).astype(bool) if k < 3 else np.frombuffer(a, np.uint8)
                          for k, a in enumerate((a_prol, a_uf, a_ul, a_nph)))
t0, t1, pb = (np.frombuffer(a, np.float64) for a in (a_t0, a_t1, a_pb))
ug, sd, frm = (np.frombuffer(a, dt) for a, dt in ((a_ug, np.int64), (a_sd, np.int32), (a_frm, np.int32)))
brk = np.frombuffer(a_brk, np.uint8).astype(bool)         # markup carries a [completion]
n_tok, n_sides_d = len(lab), len(D_SIDE)
is_speech = knd != KND["NOISE"]                       # NOISE-kind rows are [noise]/[laughter] markers, not words
n_speech = int(is_speech.sum())
dconv = np.array(D_CONV)
d_split_side = np.where((dconv >= 4010) & (dconv <= 4153), "test", np.where((dconv >= 4519) & (dconv <= 4936), "dev", "train"))
d_split = d_split_side[sd]
d_dur = np.clip(t1 - t0, 0, None)
same_prev = np.r_[False, ug[1:] == ug[:-1]]           # token continues the previous token's utterance
FORM_ID = FORM_CODE
print(f"{PROC.name}: {n_sides_d:,} sides, {n_tok:,} tokens ({n_speech:,} speech tokens + {n_tok - n_speech:,} noise/laughter markers), "
      f"{len(np.unique(ug)):,} utterances, {len(FORM_LIST):,} distinct forms")
assert n_speech == n_lex, "dataset speech tokens should equal the raw archive's lexical tokens"

### Coverage of the RSML tag set

For every RSML tag and the mispronunciation notation: what Switchboard form stands for it, which dataset label carries it, whether the label is
gold or silver, and how many tokens there are. The `construct type` column keeps `[tags]` inventory counts
separate from the `[spoken](intended)` notation. Counts follow the primary label; overlapping raw mispronunciation
markup is broken down in section 7.

In [ ]:
GOLD, SILVER, NONE_ = "gold", "silver (rule)", "not annotated in Switchboard"


def n_form(label, forms):
    ids = [FORM_ID[x] for x in forms if x in FORM_ID]
    return int(((lab == LAB[label]) & np.isin(frm, ids)).sum()) if ids else 0


H = "FILLER_HESITATION"
BC_GOLD = ["uh-huh", "um-hum", "huh-uh", "hum-um", "uh-hum"]
TAX = [
    ("hesitations", "@uhh", "uh, uhh", H, ["uh", "uhh"], GOLD),
    ("hesitations", "@umm", "um, umm", H, ["um", "umm"], GOLD),
    ("hesitations", "@hmm", "hm, hmm", H, ["hm", "hmm"], GOLD),
    ("hesitations", "@ehh", "eh, ehh", H, ["eh", "ehh"], GOLD),
    ("hesitations", "@huh", "huh", H, ["huh"], GOLD),
    ("hesitations", "@ooh", "ooh", H, ["ooh"], SILVER),
    ("hesitations", "@ohh", "ohh (plain oh is not labelled)", H, ["ohh"], SILVER),
    ("hesitations", "@uhh-oh", "uh-oh, uhh-oh", H, ["uh-oh", "uhh-oh"], SILVER),
    ("hesitations", "@uh-huh", ", ".join(BC_GOLD), "FILLER_BACKCHANNEL", BC_GOLD, GOLD),
    ("hesitations", "@ugh", "ugh", H, ["ugh"], SILVER),
    ("hesitations", "@tsk", "explicit click/tsk markers or labelled tsk/tut lexemes; see raw audit", H, TSK_LEXEMES, SILVER),
    ("hesitations", "@ho", "ho", H, ["ho"], SILVER),
    ("disfluencySpans", "filler", "discourse markers (you know, i mean, well, like, so ...)", "FILLER_DM", None, SILVER),
    ("disfluencySpans", "repetition", "adjacent repeated n-gram", "REPETITION", None, SILVER),
    ("disfluencySpans", "broken-word", "partial word x-, x[yz]-", "PARTIAL_WORD", None, GOLD),
    ("disfluencySpans", "repair", "reparandum + correction", "REPAIR", None, SILVER),
    ("disfluencySpans", "false-start", "abandoned clause start", "FALSE_START", None, SILVER),
    ("disfluencySpans", "prolongation", "unusually long function word", "PROLONGATION", None, SILVER),
    ("isolatedParalinguistics", "@laughter", "[laughter]", "NOISE", ["[laughter]"], GOLD),
]
for tag in RSML_TAGS["isolatedParalinguistics"][1:]:
    TAX.append(("isolatedParalinguistics", tag, "only generic [noise] / [vocalized-noise]", None, None, NONE_))
TAX += [
    ("isolatedOther", "@unintelligible", "-", None, None, NONE_),
    ("isolatedOther", "@stutter-block", "-", None, None, NONE_),
    ("isolatedOther", "@short-pause", "[silence] rows exist, no short/long split marked", None, None, NONE_),
    ("isolatedOther", "@long-pause", "[silence] rows exist, no short/long split marked", None, None, NONE_),
    ("paralinguisticSpans", "laughing", "[laughter-word], a word spoken while laughing", "(kind NOISE_WORD)", None, GOLD),
]
for tag in RSML_TAGS["paralinguisticSpans"]:
    if tag != "laughing":
        TAX.append(("paralinguisticSpans", tag, "-", None, None, NONE_))
for tag in RSML_TAGS["prosodySpans"]:
    TAX.append(("prosodySpans", tag, "-", None, None, NONE_))

TAX.append(("mispronunciations", RSML_MISPRON, "[spoken/intended] (non-cut-off primary labels)",
            "MISPRONUNCIATION", None, GOLD))

rows = []
for group, tag, equiv, label, forms, prov in TAX:
    if label == "(kind NOISE_WORD)":
        n = int((knd == KND["NOISE_WORD"]).sum())
    elif label is None:
        n = 0
    elif forms is None:
        n = int((lab == LAB[label]).sum())
    else:
        n = n_form(label, forms)
    if tag == "@tsk":
        # Raw explicit markers may be absent from the legacy processed labels.
        n += n_tsk_markers
        if n_tsk_markers:
            label = "FILLER_HESITATION / explicit raw marker"
            prov = "gold marker + silver lexeme" if n > n_tsk_markers else GOLD
        if n == 0:
            prov = NONE_
    rows.append({"RSML group": group, "RSML tag / notation": tag,
                 "construct type": "notation" if group == "mispronunciations" else "tag", "Switchboard equivalent": equiv,
                 "dataset label": label or "-", "provenance": prov, "tokens": n})
coverage = pd.DataFrame(rows)
for group, tags in RSML_TAGS.items():
    assert set(coverage.loc[coverage["RSML group"] == group, "RSML tag / notation"]) == set(tags)
assert coverage.loc[coverage["construct type"] == "notation", "tokens"].sum() == int((lab == LAB["MISPRONUNCIATION"]).sum())
coverage

In [ ]:
cov_sum = coverage.assign(present=coverage.tokens > 0).groupby("RSML group").agg(
    tags=("construct type", lambda s: (s == "tag").sum()),
    notations=("construct type", lambda s: (s == "notation").sum()), present=("present", "sum"),
    gold=("provenance", lambda s: (s == GOLD).sum()), silver=("provenance", lambda s: (s == SILVER).sum()),
    not_annotated=("provenance", lambda s: (s == NONE_).sum()), tokens=("tokens", "sum"))
cov_sum.loc["total"] = cov_sum.sum()
print("'present' = at least one token in the dataset. Tags listed as not annotated have no identifiable marker in this ISIP release. Mispronunciation is a notation, not a named tag.")
cov_sum

**Dataset tokens with no mapped RSML tag or notation.** Switchboard-specific types, kept visible so nothing is hidden.
Mispronunciations are now covered by `[spoken](intended)` above and are not counted as unmapped.

In [ ]:
bc_silver = int(((lab == LAB["FILLER_BACKCHANNEL"]) & ~np.isin(frm, [FORM_ID[x] for x in BC_GOLD if x in FORM_ID])).sum())
outside = pd.Series({
    "backchannel words as short turns (yeah, right, okay ...), silver": bc_silver,
    "hesitation `ah` (explicit ISIP lexeme, no RSML tag)": n_form(H, ["ah"]),
    "[noise] (generic noise)": n_form("NOISE", ["[noise]"]),
    "[vocalized-noise] (generic vocal noise)": n_form("NOISE", ["[vocalized-noise]"]),
    "coinages {word}": int((knd == KND["COINAGE"]).sum()),
}).to_frame("tokens")
outside["per 100 speech tokens"] = outside.tokens / n_speech * 100
outside

## 7. Full statistics per RSML category

Rates are **per 100 speech tokens** (words, partial words, mispronunciations and coinages; the `[noise]`,
`[vocalized-noise]` and `[laughter]` markers are not counted in the denominator). A *run* is a maximal stretch of
consecutive tokens with the same label inside one utterance, so it is the closest thing to an episode that the
per-token labels allow (two touching episodes of the same label merge into one run).

In [ ]:
PROV = {"FILLER_HESITATION": "gold lexemes + silver list", "FILLER_BACKCHANNEL": "gold hyphenated + silver short turns",
        "FILLER_DM": "silver", "REPETITION": "silver", "REPAIR": "silver", "FALSE_START": "silver",
        "PARTIAL_WORD": "gold", "PROLONGATION": "silver", "MISPRONUNCIATION": "gold", "NOISE": "gold"}
RSML_OF = {"FILLER_HESITATION": "hesitations", "FILLER_BACKCHANNEL": "hesitations (@uh-huh) / none", "FILLER_DM": "filler",
           "REPETITION": "repetition", "REPAIR": "repair", "FALSE_START": "false-start", "PARTIAL_WORD": "broken-word",
           "PROLONGATION": "prolongation", "MISPRONUNCIATION": RSML_MISPRON, "NOISE": "isolatedParalinguistics / other"}
CATS = LABELS[1:]


def runs_of(mask):
    """Maximal runs of True inside one utterance -> (start indices, lengths)."""
    prev = np.r_[False, mask[:-1]] & same_prev
    starts = mask & ~prev
    run_id = np.cumsum(starts)
    return np.flatnonzero(starts), np.bincount(run_id[mask])[1:]


n_utt_d = np.unique(ug[is_speech]).size
rows = {}
for name in CATS:
    m = lab == LAB[name]
    st, ln = runs_of(m)
    rows[name] = {
        "RSML tag": RSML_OF[name], "provenance": PROV[name], "tokens": int(m.sum()),
        "per 100 speech tokens": m.sum() / n_speech * 100,
        "runs": len(ln), "mean run length": ln.mean(),
        "% of sides with 1+": np.unique(sd[m]).size / n_sides_d * 100,
        "% of utterances with 1+": np.unique(ug[m]).size / n_utt_d * 100,
    }
cat_tab = pd.DataFrame(rows).T
cat_tab

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = cat_tab.sort_values("per 100 speech tokens")
colors = [BLUE if p.startswith("gold") else (ORANGE if p == "silver" else GREEN) for p in order.provenance]
ax.barh(order.index, order["per 100 speech tokens"].astype(float), color=colors)
for i, v in enumerate(order["per 100 speech tokens"].astype(float)):
    ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
ax.set_xlabel("tokens per 100 speech tokens")
ax.set_title("Labelled categories (blue = gold, orange = silver, green = mixed)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()
silver_share = cat_tab.loc[[c for c in GAP_LABELS + ["PROLONGATION"] if PROV[c] == "silver"], "tokens"].sum() / cat_tab.loc[GAP_LABELS + ["PROLONGATION"], "tokens"].sum()
print(f"Of the {int(cat_tab.loc[GAP_LABELS + ['PROLONGATION'], 'tokens'].sum()):,} disfluency tokens (removable categories + prolongation), "
      f"{pct(silver_share, 1)} sit in purely silver categories.")

### Special token types x label

How the dataset's token kinds (`kind`) fall on the labels. `NOISE_WORD` is `[laughter-word]`.

In [ ]:
pd.crosstab(pd.Categorical.from_codes(knd, KINDS), pd.Categorical.from_codes(lab, LABELS),
            rownames=["kind"], colnames=["label"])

### Roles: reparandum, interregnum, repair

In [ ]:
pd.crosstab(pd.Categorical.from_codes(lab, LABELS), pd.Categorical.from_codes(rol, ["(none)"] + ROLES[1:]),
            rownames=["label"], colnames=["role"]).replace(0, np.nan)

### What each category is made of

Most frequent surface forms, and for the run-based categories the most frequent phrases.

In [ ]:
def top_forms(name, k=8):
    m = lab == LAB[name]
    c = np.bincount(frm[m], minlength=len(FORM_LIST))
    top = np.argsort(-c)[:k]
    return ",  ".join(f"{FORM_LIST[i]} {c[i]:,}" for i in top if c[i])


def top_phrases(name, k=10):
    st, ln = runs_of(lab == LAB[name])
    cnt = Counter(" ".join(FORM_LIST[x] for x in frm[s:s + n]) for s, n in zip(st, ln))
    tot = sum(cnt.values())
    return ",  ".join(f"{p} {c:,}" for p, c in cnt.most_common(k)), cnt


pd.set_option("display.max_colwidth", 220)
forms_tab = pd.DataFrame({"most frequent forms": {n: top_forms(n) for n in CATS if n != "NOISE"}})
forms_tab.loc["NOISE"] = top_forms("NOISE", 4)
forms_tab

In [ ]:
phr = {n: top_phrases(n, 10)[0] for n in ["FILLER_HESITATION", "FILLER_BACKCHANNEL", "FILLER_DM", "REPETITION", "REPAIR", "FALSE_START"]}
pd.DataFrame({"most frequent runs (phrase, count)": phr})

### Run-length structure

How long the episodes are, for the categories that can span several tokens.

In [ ]:
SPAN_CATS = ["FILLER_HESITATION", "FILLER_BACKCHANNEL", "FILLER_DM", "REPETITION", "REPAIR", "FALSE_START"]
len_rows, len_data = {}, {}
for name in SPAN_CATS:
    st, ln = runs_of(lab == LAB[name])
    len_data[name] = ln
    b = np.minimum(ln, 6)
    r = {f"{k}" if k < 6 else "6+": (b == k).mean() for k in range(1, 7)}
    r.update({"runs": len(ln), "mean": ln.mean(), "median": float(np.median(ln)), "max": int(ln.max())})
    len_rows[name] = r
pd.DataFrame(len_rows).T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5.2), sharey=True)
for ax, name in zip(axes.ravel(), SPAN_CATS):
    b = np.minimum(len_data[name], 8)
    cnt = np.bincount(b, minlength=9)[1:] / len(b) * 100
    ax.bar([str(k) if k < 8 else "8+" for k in range(1, 9)], cnt, color=BLUE)
    ax.set_title(name)
    ax.set_xlabel("tokens in the run")
axes[0, 0].set_ylabel("% of runs")
axes[1, 0].set_ylabel("% of runs")
plt.tight_layout()
plt.show()

### Repair episodes: reparandum, interregnum, correction

For every `REPAIR` run: how long is the discarded material, is there an editing term between it and the
correction, and how long is the correction (`role=repair`)?

In [ ]:
st, ln = runs_of(lab == LAB["REPAIR"])
rep_rows = []
for s, l in zip(st, ln):
    j, inter = s + l, 0
    while j < n_tok and same_prev[j] and rol[j] == ROL["interregnum"]:
        inter += 1
        j += 1
    corr = 0
    while j < n_tok and same_prev[j] and rol[j] == ROL["repair"]:
        corr += 1
        j += 1
    rep_rows.append((l, inter, corr))
rep = pd.DataFrame(rep_rows, columns=["reparandum", "interregnum", "correction"])
repair_summary = pd.DataFrame({
    "episodes": [len(rep)] * 3,
    "mean length": [rep.reparandum.mean(), rep.interregnum.mean(), rep.correction.mean()],
    "median": [rep.reparandum.median(), rep.interregnum.median(), rep.correction.median()],
    "share with length 0": [(rep.reparandum == 0).mean(), (rep.interregnum == 0).mean(), (rep.correction == 0).mean()],
    "max": [rep.reparandum.max(), rep.interregnum.max(), rep.correction.max()],
}, index=["reparandum", "interregnum (editing term)", "correction"])
print(f"{(rep.interregnum > 0).mean() * 100:.1f}% of repair episodes have an editing term between reparandum and correction; "
      f"{(rep.correction > 0).mean() * 100:.1f}% have a labelled correction run right after")
repair_summary

### Partial words (`broken-word`)

In [ ]:
pidx = np.flatnonzero(lab == LAB["PARTIAL_WORD"])
cut_tab = pd.Series({"fragment ends the word, th[e]- / i-": int((cut_t[pidx] == CUT["end"]).sum()),
                     "fragment starts the word, -[o]kay": int((cut_t[pidx] == CUT["start"]).sum()),
                     "cut on both sides, -[a]b[out]-": int((cut_t[pidx] == CUT["both"]).sum())}).to_frame("tokens")
cut_tab["share"] = cut_tab.tokens / cut_tab.tokens.sum()
print(f"{pct(brk[pidx].mean(), 1)} of partial words carry an annotated completion in brackets (the rest are bare, like `i-`)")

res = Counter()
for i in pidx:
    want = intended_of.get(int(i), "")
    if cut_t[i] != CUT["end"] or not brk[i]:
        continue
    j = i + 1
    while j < n_tok and same_prev[j] and lab[j] in (LAB["FILLER_HESITATION"], LAB["FILLER_DM"], LAB["FILLER_BACKCHANNEL"], LAB["NOISE"]):
        j += 1
    if j >= n_tok or not same_prev[j]:
        res["partial word ends the utterance"] += 1
    elif FORM_LIST[frm[j]] == want:
        res["the intended word follows"] += 1
    elif FORM_LIST[frm[j]].startswith(FORM_LIST[frm[i]]):
        res["a word starting with the fragment follows"] += 1
    else:
        res["something else follows"] += 1
follow = pd.Series(res).to_frame("annotated end-cut partial words")
follow["share"] = follow.iloc[:, 0] / follow.iloc[:, 0].sum()
display(cut_tab)
follow

### Mispronunciations: `[spoken](intended)`

The Switchboard source stays `[spoken/intended]`; the revised RSML representation is `[spoken](intended)`.
Counts below distinguish **all raw mispronunciation pairs** (`kind=MISPRON`) from the **primary
`MISPRONUNCIATION` label**. A cut-off mispronunciation can instead have primary label `PARTIAL_WORD`;
that token is counted once in the primary-category tables and remains removable under the existing skeleton rule.


In [ ]:
mispron_by_label = Counter()
for (raw, spoken, intended, label), count in mispron_pairs.items():
    mispron_by_label[label] += count
mispron_summary = pd.Series(mispron_by_label, name="raw MISPRON tokens").to_frame()
mispron_summary["share of raw MISPRON tokens"] = mispron_summary.iloc[:, 0] / sum(mispron_by_label.values())
assert sum(mispron_by_label.values()) == int((knd == KND["MISPRON"]).sum())
assert mispron_by_label["MISPRONUNCIATION"] == int((lab == LAB["MISPRONUNCIATION"]).sum())
display(mispron_summary)
mispron_examples = pd.DataFrame([
    {"Switchboard source": raw, "RSML representation": f"[{spoken}]({intended})",
     "spoken": spoken, "intended": intended, "primary label": label, "tokens": count}
    for (raw, spoken, intended, label), count in mispron_pairs.most_common()
])
# Keep all pairs in a named table for export; show the 20 most frequent here.
mispron_examples.head(20)


### Prolongation

Prolonged words keep their place in the skeleton. For the most frequent prolonged words, how much longer are
the labelled occurrences than the unlabelled ones?

In [ ]:
pm = lab == LAB["PROLONGATION"]
c = np.bincount(frm[pm], minlength=len(FORM_LIST))
rows = {}
for i in np.argsort(-c)[:10]:
    same = frm == i
    rows[FORM_LIST[i]] = {"labelled PROLONGATION": int(c[i]),
                          "share of this word": c[i] / (same & is_speech).sum(),
                          "median duration labelled (ms)": np.median(d_dur[same & pm]) * 1000,
                          "median duration other (ms)": np.median(d_dur[same & ~pm & (lab == LAB["O"])]) * 1000}
pro = pd.DataFrame(rows).T
pro["duration ratio"] = pro["median duration labelled (ms)"] / pro["median duration other (ms)"]
print(f"'prolonged' flag (duration-based, set independently of the label): {int(prolonged.sum()):,} tokens; "
      f"{int((prolonged & (lab == LAB['PROLONGATION'])).sum()):,} of them carry the PROLONGATION label")
pro

## 8. Across categories

### Train / dev / test rates

In [ ]:
rows = {}
for name in ["train", "dev", "test", "all"]:
    m = np.ones(n_tok, bool) if name == "all" else d_split == name
    ns = (m & is_speech).sum()
    rows[name] = {"speech tokens": int(ns), **{c: (m & (lab == LAB[c])).sum() / ns * 100 for c in CATS}}
split_rates = pd.DataFrame(rows).T
split_rates

### Across speaker sides

One side is one speaker in one conversation (the release has no speaker ids). Sides with fewer than 300 speech
tokens are excluded from the per-side rates.

In [ ]:
side_speech = np.bincount(sd[is_speech], minlength=n_sides_d)
keep = side_speech >= 300
per_side = pd.DataFrame({c: np.bincount(sd[lab == LAB[c]], minlength=n_sides_d)[keep] / side_speech[keep] * 100 for c in CATS})
q = per_side.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).T
q["mean"] = per_side.mean()
q["max"] = per_side.max()
print(f"{keep.sum():,} of {n_sides_d:,} sides kept; rates per 100 speech tokens")
q

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 5.2))
cmap = [BLUE, ORANGE, GREEN, RED, PURPLE, GREY, BLUE, ORANGE, GREEN, RED]
for ax, c, col in zip(axes.ravel(), CATS, cmap):
    ax.hist(per_side[c], bins=35, color=col)
    ax.axvline(per_side[c].median(), color="black", lw=1, ls="--")
    ax.set_title(c, fontsize=9)
    ax.set_xlabel("per 100 speech tokens", fontsize=8)
axes[0, 0].set_ylabel("sides")
axes[1, 0].set_ylabel("sides")
plt.tight_layout()
plt.show()

per_side.corr(method="spearman")

### Position inside the utterance

Utterances with at least 3 speech tokens, so that first / middle / last are distinct positions. `NOISE` is not
positioned.

In [ ]:
utt_speech_n = np.bincount(ug[is_speech] - ug.min())            # speech tokens per utterance (index = ug - ug.min())
ok = is_speech & (utt_speech_n[ug - ug.min()] >= 3)
where = np.where(uf, "first", np.where(ul, "last", "middle"))
pos = pd.DataFrame({c: [(lab == LAB[c])[ok & (where == w)].mean() * 100 for w in ["first", "middle", "last"]]
                    for c in CATS if c != "NOISE"}, index=["first", "middle", "last"]).T

fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(pos))
for k, (w, col) in enumerate(zip(["first", "middle", "last"], [BLUE, GREY, ORANGE])):
    ax.bar(x + (k - 1) * 0.27, pos[w], 0.27, label=w, color=col)
ax.set_xticks(x, pos.index, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("per 100 tokens at that position")
ax.set_title("Where each category sits in an utterance")
ax.legend(ncol=3)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
pos

### What follows what

Row = label of a token, column = label of the next token in the same utterance (probabilities).

In [ ]:
a = lab[:-1][same_prev[1:]]
b = lab[1:][same_prev[1:]]
M = np.bincount(a.astype(int) * 11 + b, minlength=121).reshape(11, 11).astype(float)
P = M / M.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8.6, 6.4))
im = ax.imshow(P, cmap="Blues", vmin=0, vmax=0.6)
ax.set_xticks(range(11), LABELS, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(11), LABELS, fontsize=8)
for i in range(11):
    for j in range(11):
        if P[i, j] >= 0.01:
            ax.text(j, i, f"{P[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if P[i, j] > 0.35 else "black")
ax.set_xlabel("next token")
ax.set_ylabel("this token")
ax.set_title("Label transition probabilities")
ax.grid(False)
plt.tight_layout()
plt.show()

## 9. Timing: pauses and durations per category

*Pause before* is the silence directly before a token (from the dataset); utterance-initial tokens are excluded
because the gap there is a turn change, not a hesitation.

In [ ]:
rows = {}
for name in LABELS:
    m = (lab == LAB[name]) & (is_speech | (name == "NOISE"))
    p = pb[m & ~uf]
    pos_p = p[p > 1e-6]
    rows[name] = {"tokens": int(m.sum()),
                  "median duration (ms)": np.median(d_dur[m]) * 1000,
                  "pause before > 0": (p > 1e-6).mean(),
                  "median pause when > 0 (ms)": np.median(pos_p) * 1000 if len(pos_p) else np.nan,
                  "pause > 250 ms": (p > 0.25).mean(),
                  "median phones": float(np.median(nph[m & (nph > 0)])) if (m & (nph > 0)).any() else np.nan}
timing = pd.DataFrame(rows).T
timing

In [ ]:
def ecdf(ax, v, label, col):
    v = np.sort(v)
    ax.plot(v, np.arange(1, len(v) + 1) / len(v), label=label, color=col, lw=1.5)


tab10 = plt.get_cmap("tab10").colors
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k, name in enumerate([l for l in LABELS if l not in ("NOISE", "MISPRONUNCIATION")]):
    m = (lab == LAB[name]) & is_speech
    p = pb[m & ~uf]
    p = p[p > 1e-6]
    ecdf(axes[0], p, name, tab10[k] if name != "O" else GREY)
    d = d_dur[m]
    ecdf(axes[1], d[d > 0], name, tab10[k] if name != "O" else GREY)
axes[0].set_xscale("log")
axes[0].set_xlabel("pause before the token, seconds (only pauses > 0)")
axes[0].set_ylabel("cumulative share")
axes[0].set_title("Pause before")
axes[0].legend(fontsize=7, loc="lower right")
axes[1].set_xlim(0, 1.2)
axes[1].set_xlabel("token duration, seconds")
axes[1].set_title("Duration")
plt.tight_layout()
plt.show()

## 10. The fluent skeleton: what removing the disfluencies leaves

The skeleton keeps tokens labelled `O`, `PROLONGATION` and `MISPRONUNCIATION`; the seven removable categories
(`FILLER_*`, `REPETITION`, `REPAIR`, `FALSE_START`, `PARTIAL_WORD`) are deleted, with `[noise]` markers ignored.
A **gap** is a maximal run of removed tokens inside one utterance.

In [ ]:
k = is_speech
lab_k, ug_k = lab[k], ug[k]
same_k = np.r_[False, ug_k[1:] == ug_k[:-1]]
ustart_k = np.flatnonzero(np.r_[True, ug_k[1:] != ug_k[:-1]])
usize_k = np.diff(np.r_[ustart_k, len(ug_k)])


def skeleton(labels):
    """Delete the tokens carrying `labels`; return per-token mask, gap ids and summary numbers."""
    removed = np.isin(lab_k, [LAB[x] for x in labels])
    prev_rem = np.r_[False, removed[:-1]] & same_k
    gap_start = removed & ~prev_rem
    gap_id = np.cumsum(gap_start)
    gap_len = np.bincount(gap_id[removed])[1:]
    urem = np.add.reduceat(removed.astype(np.int64), ustart_k)
    fluent_n = usize_k - urem
    changed = urem > 0
    return removed, gap_start, gap_id, gap_len, urem, fluent_n, changed


VARIANTS = {
    "all 7 removable labels (dataset definition)": GAP_LABELS,
    "without FILLER_BACKCHANNEL": [x for x in GAP_LABELS if x != "FILLER_BACKCHANNEL"],
    "without FILLER_BACKCHANNEL and FILLER_DM": [x for x in GAP_LABELS if x not in ("FILLER_BACKCHANNEL", "FILLER_DM")],
}
rows = {}
for name, labels in VARIANTS.items():
    removed, gap_start, gap_id, gap_len, urem, fluent_n, changed = skeleton(labels)
    rows[name] = {
        "tokens removed": int(removed.sum()), "% of speech tokens": removed.mean() * 100,
        "utterances changed (%)": changed.mean() * 100, "utterances left empty": int((fluent_n == 0).sum()),
        "gaps": len(gap_len), "mean gap length": gap_len.mean(), "gaps that are 1 token (%)": (gap_len == 1).mean() * 100,
        "gaps per 100 skeleton tokens": len(gap_len) / fluent_n.sum() * 100,
        "mean tokens per utterance, fluent": fluent_n.mean(),
    }
effect = pd.DataFrame(rows).T
print(f"original: {len(usize_k):,} utterances with speech tokens, {usize_k.mean():.2f} tokens per utterance on average. "
      "Backchannel turns (yeah, uh-huh) are whole utterances, which is why many utterances become empty when they are removed.")
effect

In [ ]:
# the dataset's own definition (all 7 labels) is used from here on
removed, gap_start, gap_id, gap_len, urem, fluent_n, changed = skeleton(GAP_LABELS)
n_gap = len(gap_len)

In [ ]:
# composition of the gaps: which labels they contain
has = np.zeros((n_gap, len(GAP_LABELS)), bool)
for j, name in enumerate(GAP_LABELS):
    sel = removed & (lab_k == LAB[name])
    has[:, j] = np.bincount(gap_id[sel], minlength=n_gap + 1)[1:] > 0
key = (has * (1 << np.arange(len(GAP_LABELS)))).sum(axis=1)
short = {"FILLER_HESITATION": "HES", "FILLER_BACKCHANNEL": "BC", "FILLER_DM": "DM", "REPETITION": "REP",
         "REPAIR": "RPR", "FALSE_START": "FS", "PARTIAL_WORD": "PART"}
names = {kk: "+".join(short[GAP_LABELS[j]] for j in range(len(GAP_LABELS)) if kk >> j & 1) for kk in np.unique(key)}
comp = pd.Series(key).map(names).value_counts()
comp_tab = comp.head(14).to_frame("gaps")
comp_tab["share"] = comp_tab.gaps / n_gap
comp_tab["mean gap length"] = [gap_len[pd.Series(key).map(names).to_numpy() == n].mean() for n in comp_tab.index]
print(f"{len(comp)} distinct label combinations; the 14 most frequent cover {pct(comp.head(14).sum() / n_gap, 1)} of the gaps "
      f"(HES hesitation, BC backchannel, DM discourse marker, REP repetition, RPR repair, FS false start, PART partial word)")
comp_tab

In [ ]:
gl = pd.Series(np.minimum(gap_len, 6)).value_counts().sort_index()
gl.index = [str(x) if x < 6 else "6+" for x in gl.index]
per_utt = pd.Series(np.minimum(np.add.reduceat(gap_start.astype(np.int64), ustart_k)[changed], 4)).value_counts().sort_index()
per_utt.index = [str(x) if x < 4 else "4+" for x in per_utt.index]
bucket_n = pd.cut(usize_k, bins=[0, 1, 2, 4, 8, 16, 32, 10_000], labels=["1", "2", "3-4", "5-8", "9-16", "17-32", "33+"])
p_changed = pd.Series(changed).groupby(bucket_n, observed=True).mean() * 100

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].bar(gl.index, gl.to_numpy() / gl.sum() * 100, color=BLUE)
axes[0].set_xlabel("tokens in the gap")
axes[0].set_ylabel("% of gaps")
axes[0].set_title("Gap length")
axes[1].bar(per_utt.index, per_utt.to_numpy() / per_utt.sum() * 100, color=ORANGE)
axes[1].set_xlabel("gaps in the utterance")
axes[1].set_ylabel("% of changed utterances")
axes[1].set_title("Gaps per changed utterance")
axes[2].plot(p_changed.index.astype(str), p_changed.to_numpy(), "o-", color=RED)
axes[2].set_xlabel("speech tokens in the utterance")
axes[2].set_ylabel("% of utterances with 1+ removed token")
axes[2].set_title("Longer utterances are changed more often")
plt.tight_layout()
plt.show()

## 11. Headline numbers

In [ ]:
headline = pd.Series({
    "conversations": len(set(conv)),
    "speaker sides": len(SIDES),
    "hours of conversation audio": side_dur.sum() / 3600 / 2,
    "speech tokens": n_speech,
    "word types": len(vocab),
    "RSML tags in the taxonomy": int((coverage["construct type"] == "tag").sum()),
    "RSML tags with 1+ token in the dataset": int(((coverage["construct type"] == "tag") & (coverage.tokens > 0)).sum()),
    "RSML notations covered": int((coverage["construct type"] == "notation").sum()),
    "mispronunciation notation": RSML_MISPRON,
    "explicit raw click/tsk markers": n_tsk_markers,
    **{f"{c} per 100 speech tokens": cat_tab.loc[c, "per 100 speech tokens"] for c in CATS},
    "removable tokens per 100 speech tokens": removed.mean() * 100,
    "utterances changed by removal": changed.mean(),
    "mean gap length (tokens)": gap_len.mean(),
    "share of gaps that are exactly 1 token": (gap_len == 1).mean(),
}, dtype=object).to_frame("value")
headline

### Notes and limits

* **Silver labels are heuristics.** `REPETITION`, `REPAIR`, `FALSE_START`, `PROLONGATION`, `FILLER_DM` and the
  short-turn backchannels come from rules, not from human annotation. Their counts describe what the rules fire
  on and carry unknown precision and recall; the gold categories (`PARTIAL_WORD`, `MISPRONUNCIATION`, `NOISE`,
  the hesitation lexemes) are explicit in the ISIP markup.
* Mispronunciations map to `[spoken](intended)`. Raw mispronunciation markup on a cut-off word can have
  primary label `PARTIAL_WORD`; these overlaps are reported separately, not added twice to primary-label totals.
* `@tsk` requires an explicit click/tsk marker or a labelled hesitation lexeme. Ordinary lexical `click`
  is not a tongue-click annotation; generic noise markers do not identify the underlying sound.
* **Runs are not episodes.** The per-token labels cannot separate two touching episodes of the same label.
* RSML tags with no Switchboard counterpart (`@stutter-block`, `@yawn`, pitch spans, ...) are listed in the
  coverage table with 0 tokens: the corpus simply does not mark them.
* `[silence]` rows also cover stretches where the *other* speaker is talking, so a side's silence share is not a
  pause rate; use the *pause before* statistics, which only look inside an utterance.
* The split follows the conversation-id ranges used earlier in the project (test sw4010 to sw4153, dev sw4519
  to sw4936).